In [1]:
import sys, importlib.util, dataclasses
from pathlib import Path
from datetime import date
import pandas as pd

for _c in [Path.cwd()] + list(Path.cwd().parents):
    if (_c / "gbp").is_dir() and (_c / "tests").is_dir():
        sys.path.insert(0, str(_c)); break

_spec = importlib.util.spec_from_file_location("fixtures", _c / "tests/unit/build/fixtures.py")
_mod = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_mod)
minimal_raw_model = _mod.minimal_raw_model

from gbp.core.enums import FacilityRole, FacilityType, ModalType, OperationType, PeriodType, AttributeKind, ResourceStatus
from gbp.core.roles import DEFAULT_ROLES, derive_roles
from gbp.core.model import RawModelData, ResolvedModelData
from gbp.build.pipeline import _ensure_edges_and_commodities, build_model
from gbp.build.validation import validate_raw_model
from gbp.build.time_resolution import resolve_all_time_varying, resolve_registry_attributes
from gbp.build.lead_time import resolve_lead_times
from gbp.build.transformation import resolve_transformations
from gbp.build.fleet_capacity import compute_fleet_capacity
from gbp.build.spine import assemble_spines
from gbp.consumers.simulator.state import init_state
from gbp.loaders import DataLoaderMock, DataLoaderGraph, GraphLoaderConfig
from gbp.loaders.contracts import (
    DepotsSourceSchema,
    GraphLoaderConfig,
    ResourcesSourceSchema,
    StationsSourceSchema,
)
from gbp.core.attributes.registry import AttributeRegistry

In [5]:
mock_config = {
    "n": 10,               # 10 stations
    "n_depots": 2,         # 2 depots
    "n_timestamps": 168,   # 1 week of hourly data
    "time_freq": "h",
    "start_date": "2025-01-01",
    "num_resources": 3,
    "resource_capacity": 100,
    "seed": 42,
}

# Loading of mock data
mock = DataLoaderMock(mock_config)

n_stations = mock.config["n"]
n_depots = mock.config.get("n_depots", 2)
n_timestamps = mock.config.get("n_timestamps", 168)
start_date = mock.config.get("start_date", "2025-01-01")
freq = mock.config.get("time_freq", "h")

mock.df_stations = mock._generate_stations(n_stations)
mock.df_depots = mock._generate_depots(n_depots)
mock.df_resources = mock._generate_resources()
mock.timestamps = pd.date_range(start=start_date, periods=n_timestamps, freq=freq)
mock.df_inventory_ts, mock.df_telemetry_ts, mock.df_trips = mock._generate_telemetry_and_trips(
    df_stations=mock.df_stations,
    timestamps=mock.timestamps,
)
mock.df_station_costs, mock.df_truck_rates = mock._generate_costs(
    df_stations=mock.df_stations,
    df_resources=mock.df_resources,
)

# Loading of RawModelData using DataLoaderGraph based on the mock data
loader = DataLoaderGraph(mock, GraphLoaderConfig())

StationsSourceSchema.validate(loader._source.df_stations)
DepotsSourceSchema.validate(loader._source.df_depots)
ResourcesSourceSchema.validate(loader._source.df_resources)

temporal = loader._build_temporal()
entities = loader._build_entities()
behavior = loader._build_behavior(entities)
edge_data = loader._build_edges(entities) if loader._config.build_edges else {}
resources = loader._build_resources(entities)
registry = AttributeRegistry()
flow_data = loader._build_node_parameters(entities, registry)
loader._register_costs(registry, temporal)
loader._register_resource_costs(registry, entities)
all_tables = {
    **temporal,
    **entities.tables,
    **behavior,
    **edge_data,
    **flow_data,
    **resources,
}
loader._raw = RawModelData(
        **{k: v for k, v in all_tables.items() if v is not None},
        attributes=registry,
    )


# Validation of RawModelData 
loader._raw.validate()
validate_raw_model(loader._raw).raise_if_invalid()
periods = loader._raw.periods.copy()
resolved_time = resolve_all_time_varying(loader._raw, periods)
resolved_attrs = resolve_registry_attributes(loader._raw.attributes, periods)
edges_df, ec_df = _ensure_edges_and_commodities(loader._raw)
edge_lead_time_resolved: pd.DataFrame | None = None
if edges_df is not None and not edges_df.empty:
    elt = resolve_lead_times(edges_df, periods)
    edge_lead_time_resolved = elt if not elt.empty else None
transformation_resolved = resolve_transformations(
    loader._raw.facilities,
    loader._raw.transformations,
    loader._raw.transformation_inputs,
    loader._raw.transformation_outputs,
)
fleet_capacity = compute_fleet_capacity(
    loader._raw.resource_fleet,
    loader._raw.resource_categories,
    loader._raw.resources,
)
resolved = ResolvedModelData.from_raw(
    loader._raw,
    periods=periods,
    resolved_time=resolved_time,
    resolved_attrs=resolved_attrs,
    edges=edges_df,
    edge_commodities=ec_df,
    edge_lead_time_resolved=edge_lead_time_resolved,
    transformation_resolved=transformation_resolved,
    fleet_capacity=fleet_capacity,
)
spines = assemble_spines(resolved)
resolved.facility_spines = spines["facility"] or None
resolved.edge_spines = spines["edge"] or None
resolved.resource_spines = spines["resource"] or None